In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

np.random.seed(42)

## Task 1

In [2]:
# observed counts
observed = np.array([120, 85, 110, 85])
total_n = observed.sum()
k = len(observed)

# expected counts
expected = np.full(k, total_n / k)

chi2_stat, p_value = st.chisquare(observed, f_exp=expected)
print(f"Chi-square statistics :{chi2_stat}")
print(f"p-value :{p_value}")
print(f"Degrees of freedom: {len(observed)-1}")

if p_value < 0.05:
    print("Reject Null hypothesis H_0 : players choose among four character classes in equal proportions")
else:
    print("Reject Alternative hypothesis H_1 : No strong evidence the prefernces are equal")

Chi-square statistics :9.5
p-value :0.023331360430831553
Degrees of freedom: 3
Reject Null hypothesis H_0 : players choose among four character classes in equal proportions


In [3]:
categories = ['Warrior', 'Mage', 'Rogue', 'Healer']
residuals = observed - expected

results = pd.DataFrame({
    'Category': categories,
    'Observed': observed,
    'Expected': expected,
    'Difference': residuals
    })
results

,Category,Observed,Expected,Difference
0,Warrior,120,100.0,20.0
1,Mage,85,100.0,-15.0
2,Rogue,110,100.0,10.0
3,Healer,85,100.0,-15.0


Warrior(20.0) and Rogue(10.0) classes are over-represented

Mage(15.0) and Healer(15.0) are under-represented

## Task 2

In [15]:
n = 600
subscription_tiers = np.random.choice(['Free', 'Basic', 'Premium'], size = n, p=[1/3, 1/3, 1/3])
churn_status = []
for s in subscription_tiers:
    if s=='Free':
        churn_status.append(np.random.choice(['Churned', 'Retained'], p=[0.45, 0.55]))
    elif s=='Basic':
        churn_status.append(np.random.choice(['Churned', 'Retained'], p=[0.3, 0.7]))
    else:
        churn_status.append(np.random.choice(['Churned', 'Retained'], p=[0.15, 0.85]))

df = pd.DataFrame({"subscription_tiers":subscription_tiers, "churn_status":churn_status})
contingency = pd.crosstab(df["subscription_tiers"], df["churn_status"])
print("Contingency Table:")
print(contingency)

Contingency Table:
churn_status        Churned  Retained
subscription_tiers                   
Basic                    58       132
Free                     98       106
Premium                  32       174


In [13]:
chi2, p_value, dof, expected = st.chi2_contingency(contingency)

print(f"Chi-square statistic: {chi2}")
print(f"p-value: {p_value:6f}")
print(f"Degrees of freedom: {dof}")

print(f"\nExpected frequencies :")
expected_df = pd.DataFrame(
    expected, index=contingency.index, columns=contingency.columns
)
print(expected_df.round(1))

Chi-square statistic: 49.83243972762273
p-value: 0.000000
Degrees of freedom: 2

Expected frequencies :
churn_status        Churned  Retained
subscription_tiers                   
Basic                  59.9     142.1
Free                   59.6     141.4
Premium                58.4     138.6


There is evidence of an association between subscription tier and churn, because p-value < 0.05. That's why number of churned and retained clients changes depending on subsciption tier. We see that free people have higher churn rates and premium people have higher retention rates

## Task 3

In [21]:
def cramers_v(contigency_table):
    # chi-square
    chi2 = st.chi2_contingency(contingency)[0]
    
    # total number of observations
    n = contigency_table.values.sum()

    min_dim = min(contigency_table.shape)-1
    return np.sqrt(chi2/(n*min_dim))

v = cramers_v(contingency)
print(f"Cramér's V: {v:.4f}")

if v < 0.1:
    strength = "negligible or very small"
elif v < 0.3:
    strength = "small"
elif v < 0.5:
    strength = "medium"
else:
    strength = "large"

print(f"Association strength: {strength}")
    

Cramér's V: 0.2899
Association strength: small


The relationship between subscription tier and churn is small (weak). Although the Chi-Square test confirms that an association exists, the Cramer's V value (which is near 0.28-0.29) indicates that this connection is not very powerful.

This means that while subscription tiers do have some influence on whether a customer leaves, they are not the primary driver of churn